# Evaluating the bounded panic detector on spur-notebook

This experiment evaluates the existing detector on a separate real repository. Project source is read-only and copied byte-for-byte into task-owned evidence before analysis. The notebook separates **whole-file detector results**, **source-site candidates**, **explicit mathematical models**, and **native reproductions**. A lifted or normalized fixture never substitutes for a successful analysis of the original file. Unknown coverage remains unknown.

Detector baseline: `848a29f39511cbd27091beb6c171b97e179f353a`. Target checkout: `/Users/kevintruong/project/spur-notebook`. The registered graph points to an unavailable older path, so this run uses an isolated current-source snapshot rather than that stale graph.

In [2]:
from pathlib import Path
import hashlib, json, subprocess, os, sys, datetime, collections, shutil
PROJECT = Path("/Users/kevintruong/project/spur-notebook")
ART = Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation")
ART.mkdir(parents=True, exist_ok=True)
CAPTURE = ART / "source-snapshot"
BIN = Path("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
DETECTOR_SOURCE = Path("/private/tmp/r8g-root-review-worktree")
def sha(data): return hashlib.sha256(data).hexdigest()
def save(path, value): path.write_text(json.dumps(value, indent=2) + "\n")
def git(*args): return subprocess.check_output(["git", "-C", str(PROJECT), *args])
assert PROJECT.is_dir() and BIN.is_file()
assert sha(BIN.read_bytes()) == "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd"
assert not CAPTURE.exists(), "Do not overwrite an earlier captured source set"
tracked = set(p.decode() for p in git("ls-files", "-z", "--", "*.rs", "**/AGENTS.md", "AGENTS.md", "**/Cargo.toml", "Cargo.toml", "**/Cargo.lock", "Cargo.lock", ".cargo/config.toml").split(b"\0") if p)
untracked_rust = set(p.decode() for p in git("ls-files", "--others", "--exclude-standard", "-z", "--", "*.rs").split(b"\0") if p)
source_rows, skipped = [], []
for relative in sorted(tracked | untracked_rust):
    origin = PROJECT / relative
    if not origin.is_file() or origin.is_symlink():
        skipped.append({"path":relative,"reason":"missing or symlink"}); continue
    data = origin.read_bytes()
    destination = CAPTURE / relative
    destination.parent.mkdir(parents=True,exist_ok=True)
    destination.write_bytes(data)
    assert destination.read_bytes() == data
    source_rows.append({"path":relative,"bytes":len(data),"sha256":sha(data),
                        "git_blob_oid":hashlib.sha1(b"blob "+str(len(data)).encode()+b"\0"+data).hexdigest(),
                        "tracked":relative in tracked,"rust":relative.endswith(".rs")})
project_state = {
    "root":str(PROJECT),"head":git("rev-parse","HEAD").decode().strip(),
    "status_porcelain":git("status","--porcelain=v1","--untracked-files=normal").decode(),
    "capture_created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_manifest_sha256":sha(json.dumps(source_rows,sort_keys=True,separators=(",",":")).encode()),
    "files":len(source_rows),"rust_files":sum(x["rust"] for x in source_rows),
    "source_bytes":sum(x["bytes"] for x in source_rows),
    "detector_binary_sha256":sha(BIN.read_bytes()),"skipped":skipped,
    "registered_graph_error":"spur-notebook registry points to missing /Volumes/Projects/spur-notebook; current-source fallback",
}
save(ART/"source-manifest.json",source_rows)
save(ART/"project-state-before.json",project_state)
print(json.dumps({k:v for k,v in project_state.items() if k!="status_porcelain"},indent=2))
print("Working-tree status retained locally; project files were only read.")


{
  "root": "/Users/kevintruong/project/spur-notebook",
  "head": "7fb364638575da96d46a68c5ec6af0215fe97300",
  "capture_created_utc": "2026-11-02T06:36:51.472887+00:00",
  "source_manifest_sha256": "f56a8ac766e7d8e468ca6776857a0229525e2b1170c3dff3a80c57e8e4586964",
  "files": 721,
  "rust_files": 689,
  "source_bytes": 14154913,
  "detector_binary_sha256": "984ee7bc02555b9c831addac41ce44d0bf323bf685f011c52078f815363eb5cd",
  "skipped": [],
  "registered_graph_error": "spur-notebook registry points to missing /Volumes/Projects/spur-notebook; current-source fallback"
}
Working-tree status retained locally; project files were only read.


In [3]:
# Build the actual source graph and direct method-site evidence from captured project bytes.
# This does not invoke Cargo or compile the target application.
RUN = ART / "detector"
RUN.mkdir(exist_ok=True)
ENV = os.environ.copy()
ENV.update(RUSTUP_TOOLCHAIN="1.94.1-aarch64-apple-darwin", TMPDIR="/private/tmp")
snapshot_request = {"command":"snapshot","root":str(CAPTURE),
                    "source_paths":[r["path"] for r in source_rows if r["rust"]]}
save(RUN / "project-snapshot.request.json", snapshot_request)
snapshot_stdin = open(RUN/"project-snapshot.request.json","rb")
snapshot_stdout = open(RUN/"project-snapshot.stdout","wb")
snapshot_stderr = open(RUN/"project-snapshot.stderr","wb")
snapshot_started = datetime.datetime.now(datetime.timezone.utc).isoformat()
snapshot_job = subprocess.Popen([str(BIN)],stdin=snapshot_stdin,stdout=snapshot_stdout,stderr=snapshot_stderr,
                                cwd=RUN,env=ENV)
print(json.dumps({"phase":"snapshot","pid":snapshot_job.pid,"started_utc":snapshot_started,
                  "rust_files":len(snapshot_request["source_paths"]),
                  "source_manifest_sha256":project_state["source_manifest_sha256"]},indent=2))


{
  "phase": "snapshot",
  "pid": 51905,
  "started_utc": "2026-11-02T06:39:09.542067+00:00",
  "rust_files": 689,
  "source_manifest_sha256": "f56a8ac766e7d8e468ca6776857a0229525e2b1170c3dff3a80c57e8e4586964"
}


In [4]:
snapshot_exit = snapshot_job.poll()
print(json.dumps({"snapshot_exit":snapshot_exit,"snapshot_output_bytes":(RUN/"project-snapshot.stdout").stat().st_size}))
if snapshot_exit is not None:
    assert snapshot_exit == 0, (RUN/"project-snapshot.stderr").read_text(errors="replace")[-3000:]
    snapshot_stdin.close(); snapshot_stdout.close(); snapshot_stderr.close()
    snapshot = json.loads((RUN/"project-snapshot.stdout").read_text())
    graph = snapshot["graph"]
    methods = snapshot["method_sites"]
    save(RUN/"project-snapshot.command.json",{
        "binary_sha256":sha(BIN.read_bytes()),"request_sha256":sha((RUN/"project-snapshot.request.json").read_bytes()),
        "exit":snapshot_exit,"started_utc":snapshot_started,
        "completed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "output_sha256":sha((RUN/"project-snapshot.stdout").read_bytes()),
        "environment":{"RUSTUP_TOOLCHAIN":ENV["RUSTUP_TOOLCHAIN"],"TMPDIR":ENV["TMPDIR"]},
    })
    method_summary={"graph_content_hash":graph["graph_content_hash"],"files":len(graph["files"]),
                    "symbols":len(graph["symbols"]),"edges":len(graph["edges"]),
                    "source_method_sites":len(methods["sites"]),
                    "macro_sites":len(graph["macro_invocations"]["rows"]),
                    "method_source_files":len(methods["sources"])}
    save(ART/"source-graph-summary.json",method_summary)
    print(json.dumps(method_summary,indent=2))
    print("First method-site shape:",json.dumps(methods["sites"][:1])[:2000])
    print("First macro-site shape:",json.dumps(graph["macro_invocations"]["rows"][:1])[:2000])
    compiler_version = subprocess.check_output(["rustc","-vV"],env=ENV,text=True)
    target = next(x[6:] for x in compiler_version.splitlines() if x.startswith("host: "))
    cfg_text = subprocess.check_output(["rustc","--print","cfg","--edition","2021","--target",target,
        "-C","overflow-checks=true","-C","debug-assertions=true","-C","panic=unwind"],env=ENV,text=True)
    cfg=[]
    for line in cfg_text.splitlines():
        key,sep,value=line.partition("=");cfg.append({"key":key,"value":json.loads(value) if sep else None})
    cfg.sort(key=lambda x:(x["key"],x["value"] or ""))
    model_build={"schema_version":"panic-build-config-v1","graph_content_hash":graph["graph_content_hash"],
                 "target":target,"edition":"2021","cfg":cfg,"overflow_checks":True,"debug_assertions":True,
                 "panic_strategy":"unwind","rustc_verbose":compiler_version,"codegen_flags":[],"features":[]}
    save(ART/"declared-analysis-profile.json",{
        "build":model_build,
        "assumption":"Explicit model profile, not an attestation that the multi-crate target project compiled under these settings. Feature/attribute uncertainty must remain unknown.",
    })
    all_sources=[{"file_path":r["path"],"text":(CAPTURE/r["path"]).read_text()} for r in source_rows if r["rust"]]
    whole_request={"command":"analyze","graph":graph,"sources":all_sources,"build":model_build}
    save(RUN/"project-analyze.request.json",whole_request)
    analyze_stdin=open(RUN/"project-analyze.request.json","rb")
    analyze_stdout=open(RUN/"project-analyze.stdout","wb")
    analyze_stderr=open(RUN/"project-analyze.stderr","wb")
    analyze_job=subprocess.Popen([str(BIN)],stdin=analyze_stdin,stdout=analyze_stdout,stderr=analyze_stderr,cwd=RUN,env=ENV)
    print(json.dumps({"whole_project_analyze_pid":analyze_job.pid,"request_bytes":(RUN/"project-analyze.request.json").stat().st_size}))


{"snapshot_exit": null, "snapshot_output_bytes": 0}


In [5]:
# Repository layout and inherited instructions, read from captured source.
print((CAPTURE/"Cargo.toml").read_text()[:10000])
for relative in ["src/AGENTS.md","jute-notebook/AGENTS.md","jute-notebook/src-tauri/AGENTS.md"]:
    path=CAPTURE/relative
    if path.is_file():
        print("\nINSTRUCTIONS:",relative)
        print(path.read_text()[:12000])
print("\nRust source groups:")
print(json.dumps(dict(collections.Counter(r["path"].split("/")[0] for r in source_rows if r["rust"])),indent=2))
print("\nSnapshot current status:",snapshot_job.poll())
print((RUN/"project-snapshot.stderr").read_text(errors="replace")[-2500:])


[workspace]
resolver = "2"
members = [
    ".",
    "crates/spurlab-telemetry",
    "jute-notebook/src-tauri",
    "rest-table-gateway",
]
exclude = ["rest-table-gateway-ext", "vendor/tauri-plugin-mcp-bridge"]

[workspace.package]
version = "1.6.0"
edition = "2021"
rust-version = "1.88"
license = "MIT"
repository = "https://github.com/getspur/spur-notebook"

[workspace.dependencies]
# Pinned SPUR-owned crates. Keep these on a specific commit or tag, not a
# branch, until the notebook-owned shared-crate boundary is finalized.
# Spur v1.24.0 plus prebuilt-DuckDB linkage support — ACP SDK 1.2.0 / schema 1.4.0.
# Keep embedding enabled; each application selects bundled or prebuilt DuckDB.
spur-acp = { git = "https://github.com/getspur/SpurCode.git", rev = "3127f115fe8245cd6647d67d95332ed9c3d5152c", package = "spur-acp" }
spur-analyst = { git = "https://github.com/getspur/SpurCode.git", rev = "3127f115fe8245cd6647d67d95332ed9c3d5152c", package = "spur-analyst", default-features = false }
sp

In [6]:
# Lexical triage only: these text matches are candidates, not semantic detector verdicts.
# The actual AST/method-site snapshot is running independently above.
import re
candidate_lines = []
patterns = {
    "byte_slice": re.compile(r"\[[^\]\n]*(?:\.\.|\.\.\=)[^\]\n]*\]"),
    "unwrap_expect": re.compile(r"\.(?:unwrap|expect)\s*\("),
}
owned_prefixes=("src/","jute-notebook/src-tauri/src/","rest-table-gateway/src/","crates/spurlab-telemetry/src/")
for record in source_rows:
    if not record["rust"] or not record["path"].startswith(owned_prefixes):
        continue
    lines=(CAPTURE/record["path"]).read_text().splitlines()
    for number,line in enumerate(lines,1):
        if patterns["byte_slice"].search(line) and any(token in line for token in ["&", "str", "text", "code", "query", "input", "value", "name", "output", "buffer", "source", "line"]):
            candidate_lines.append({"path":record["path"],"line":number,"text":line.strip()[:250]})
save(ART/"lexical-byte-slice-triage.json",candidate_lines)
print(json.dumps({"matches":len(candidate_lines),"sample":candidate_lines[:85]},indent=2))
print("Actual snapshot exit:",snapshot_job.poll())
try:
    process = subprocess.run(["ps","-p",str(snapshot_job.pid),"-o","pid,etime,time,rss,stat,command"],
                             capture_output=True,text=True,timeout=5)
    print(process.stdout[:2000] or process.stderr[:500])
except Exception as error:
    print("Process observation unavailable:",type(error).__name__)


{
  "matches": 185,
  "sample": [
    {
      "path": "jute-notebook/src-tauri/src/backend/local.rs",
      "line": 219,
      "text": ".args(&argv[1..])"
    },
    {
      "path": "jute-notebook/src-tauri/src/backend/notebook.rs",
      "line": 832,
      "text": "let mut remaining = &value[..];"
    },
    {
      "path": "jute-notebook/src-tauri/src/backend/notebook.rs",
      "line": 835,
      "text": "lines.push(remaining[..next_break].to_string());"
    },
    {
      "path": "jute-notebook/src-tauri/src/backend/notebook.rs",
      "line": 836,
      "text": "remaining = &remaining[next_break..];"
    },
    {
      "path": "jute-notebook/src-tauri/src/backend/wire_protocol/driver_zeromq.rs",
      "line": 55,
      "text": "let buffers = payload[delim_idx + 6..].to_vec();"
    },
    {
      "path": "jute-notebook/src-tauri/src/commands.rs",
      "line": 4199,
      "text": "let name = &sql[start..i];"
    },
    {
      "path": "jute-notebook/src-tauri/src/commands.rs",
    

In [7]:
def show(relative, first, last):
    lines=(CAPTURE/relative).read_text().splitlines()
    print("\nSOURCE",relative)
    print("\n".join(f"{number}: {lines[number-1]}" for number in range(first,min(last,len(lines))+1)))
for relative in ["rest-table-gateway/AGENTS.md","jute-notebook/src-tauri/src/AGENTS.md"]:
    if (CAPTURE/relative).is_file(): print("\nINSTRUCTIONS",relative,"\n",(CAPTURE/relative).read_text()[:10000])
show("jute-notebook/src-tauri/src/state.rs",220,310)
show("rest-table-gateway/src/adapter/manifest_adapter.rs",1,95)
show("jute-notebook/src-tauri/src/backend/wire_protocol/driver_zeromq.rs",1,95)
show("src/dag/ai/inputs.rs",535,580)
print("\nSnapshot process:",snapshot_job.poll())
print((RUN/"project-snapshot.stderr").read_text(errors="replace")[-1000:])



SOURCE jute-notebook/src-tauri/src/state.rs
220: 
221: #[derive(Debug, Serialize, Deserialize)]
222: #[serde(rename_all = "camelCase")]
223: struct StoredDatasourceEntry {
224:     #[serde(flatten)]
225:     entry: DatasourceEntry,
226:     #[serde(skip_serializing_if = "Option::is_none")]
227:     workspace_relative_path: Option<String>,
228: }
229: 
230: fn metadata_has_datasources(metadata: &NotebookMetadata) -> bool {
231:     metadata
232:         .other
233:         .get(SPUR_METADATA_KEY)
234:         .and_then(Value::as_object)
235:         .is_some_and(|spur| spur.contains_key(DATASOURCES_METADATA_KEY))
236: }
237: 
238: fn workspace_root() -> PathBuf {
239:     env::current_dir().unwrap_or_else(|_| PathBuf::from("."))
240: }
241: 
242: fn has_credential_uri(entry: &DatasourceEntry) -> bool {
243:     use crate::commands::DatasourceKind;
244: 
245:     // Only explicit credential URIs bypass filesystem handling. Collapsed
246:     // `credential:/` paths can also name real lo

In [8]:
# Exact function segments for source-faithful replay; no target source is changed.
def segment(relative, first, last):
    data=(CAPTURE/relative).read_bytes()
    lines=data.splitlines(keepends=True)
    start=sum(map(len,lines[:first-1])); end=sum(map(len,lines[:last]))
    part=data[start:end]
    return {"path":relative,"line_range":[first,last],"byte_range":[start,end],
            "file_sha256":sha(data),"segment_sha256":sha(part),"text":part.decode()}
segments = {
 "object-store-constant":segment("jute-notebook/src-tauri/src/state.rs",259,261),
 "repair-object-store":segment("jute-notebook/src-tauri/src/state.rs",263,284),
 "object-store-caller":segment("jute-notebook/src-tauri/src/state.rs",286,297),
 "header-prefix":segment("rest-table-gateway/src/adapter/manifest_adapter.rs",43,56),
 "guarded-bounded-text":segment("src/dag/ai/inputs.rs",555,568),
}
save(ART/"replay-source-segments.json",segments)
for key in ["repair-object-store","header-prefix","guarded-bounded-text"]:
    print(key, json.dumps({k:v for k,v in segments[key].items() if k!="text"}))
print("\nSource caller matches (read-only source text, no reachability conclusion):")
for name in ["repair_object_store_location","is_object_store_location","apply_header_value_prefix","bounded_text"]:
    matches=[]
    for record in source_rows:
        if record["rust"] and record["path"].startswith(owned_prefixes):
            for number,line in enumerate((CAPTURE/record["path"]).read_text().splitlines(),1):
                if name+"(" in line: matches.append({"path":record["path"],"line":number,"text":line.strip()[:220]})
    print(name,json.dumps(matches[:18],indent=2))
print("Snapshot:",snapshot_job.poll())


repair-object-store {"path": "jute-notebook/src-tauri/src/state.rs", "line_range": [263, 284], "byte_range": [8792, 9600], "file_sha256": "3bd47d2605d21fe76979c5ba295f6e8167558c6cc4a7811601f0644b4cfb5ec8", "segment_sha256": "7498532fa1a49cdd92ae81c5fe517e1c8eafeadb1eb97caad6fdced99acc02e8"}
header-prefix {"path": "rest-table-gateway/src/adapter/manifest_adapter.rs", "line_range": [43, 56], "byte_range": [1699, 2172], "file_sha256": "35cd6bef5a954004c138958ef05ead4bc0f111747c0de67f5e6f32f6436e99a2", "segment_sha256": "ca54d38ae7c7b15f3dca0b08799ca1236a10e728561bf8692eec1be20a2075af"}
guarded-bounded-text {"path": "src/dag/ai/inputs.rs", "line_range": [555, 568], "byte_range": [21078, 21507], "file_sha256": "6457120acf8be30a15a1755cceb0f5da086038d0d7c7e997ee6b1e2220d2bb13", "segment_sha256": "fc1a525616fd1dd767bcf27b28bff53827a19561e51161332f1a49f80e1654e2"}

Source caller matches (read-only source text, no reachability conclusion):
repair_object_store_location [
  {
    "path": "jute-no

bounded_text [
  {
    "path": "src/dag/ai/inputs.rs",
    "line": 318,
    "text": "let (rendered, truncated) = bounded_text(&rendered, remaining - overhead);"
  },
  {
    "path": "src/dag/ai/inputs.rs",
    "line": 555,
    "text": "fn bounded_text(text: &str, budget: usize) -> (String, bool) {"
  }
]
Snapshot: 1


In [9]:
# Preserve the real whole-snapshot failure; it is not a no-panic result.
snapshot_exit = snapshot_job.poll()
snapshot_error = (RUN/"project-snapshot.stderr").read_text(errors="replace")
save(RUN/"project-snapshot.command.json",{
    "binary_sha256":sha(BIN.read_bytes()),"request_sha256":sha((RUN/"project-snapshot.request.json").read_bytes()),
    "exit":snapshot_exit,"started_utc":snapshot_started,
    "observed_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "output_sha256":sha((RUN/"project-snapshot.stdout").read_bytes()),
    "stderr_sha256":sha((RUN/"project-snapshot.stderr").read_bytes()),
})
print(json.dumps({"whole_snapshot_exit":snapshot_exit,"error":snapshot_error[-7000:]},indent=2))
print("\nA related guarded implementation in the product crate:")
show("src/mcp/datasource/object_store.rs",95,160)
print("\nOriginal Jute call path:")
show("jute-notebook/src-tauri/src/state.rs",105,132)
print("\nOriginal header caller:")
show("rest-table-gateway/src/adapter/manifest_adapter.rs",365,400)


{
  "whole_snapshot_exit": 1,
  "error": "Error: missing macro source entry\n"
}

A related guarded implementation in the product crate:

SOURCE src/mcp/datasource/object_store.rs
95: }
96: 
97: #[cfg(feature = "datasource-introspect")]
98: impl std::error::Error for ObjectStoreProbeError {}
99: 
100: #[cfg(feature = "datasource-introspect")]
101: #[derive(Deserialize)]
102: #[serde(tag = "status", rename_all = "snake_case")]
103: enum ObjectStoreProbeEnvelope {
104:     Success {
105:         #[serde(rename = "duckdbVersion")]
106:         duckdb_version: String,
107:         columns: Vec<jute::commands::Column>,
108:     },
109:     Error {
110:         stage: String,
111:         message: String,
112:     },
113: }
114: 
115: /// Unix Path::new("s3://…") collapses to `/s3:/…` or `s3:/…`. DuckDB wants s3://.
116: pub(crate) fn repair_object_store_location(location: &str) -> String {
117:     let trimmed = location.trim();
118:     let mut value = trimmed.to_owned();
119:     for sche

In [10]:
# Per-file runs retain each complete original file at its original relative path.
# Isolation changes graph scope, never the source bytes. It avoids inventing successful
# whole-project results after the aggregate snapshot failed.
scan_helper = r'''from pathlib import Path
import collections,copy,hashlib,json,os,subprocess,time
ART=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation")
CAPTURE=ART/"source-snapshot"
BIN=Path("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
ENV=os.environ.copy();ENV.update(RUSTUP_TOOLCHAIN="1.94.1-aarch64-apple-darwin",TMPDIR="/private/tmp")
sha=lambda b:hashlib.sha256(b).hexdigest()
def save(p,v):p.write_text(json.dumps(v,indent=2)+"\n")
version=subprocess.check_output(["rustc","-vV"],env=ENV,text=True)
host=next(x[6:] for x in version.splitlines() if x.startswith("host: "))
rawcfg=subprocess.check_output(["rustc","--print","cfg","--edition","2021","--target",host,"-C","overflow-checks=true","-C","debug-assertions=true","-C","panic=unwind"],env=ENV,text=True)
atoms=[]
for line in rawcfg.splitlines():
 key,sep,value=line.partition("=");atoms.append({"key":key,"value":json.loads(value) if sep else None})
atoms.sort(key=lambda a:(a["key"],a["value"] or ""))
def build(graph):return {"schema_version":"panic-build-config-v1","graph_content_hash":graph["graph_content_hash"],"target":host,"edition":"2021","cfg":atoms,"overflow_checks":True,"debug_assertions":True,"panic_strategy":"unwind","rustc_verbose":version,"codegen_flags":[],"features":[]}
def cli(folder,name,request,timeout=120):
 data=json.dumps(request).encode();save(folder/(name+".request.json"),request)
 started=time.time()
 try:
  p=subprocess.run([str(BIN)],input=data,capture_output=True,cwd=folder,env=ENV,timeout=timeout)
  code=p.returncode;out=p.stdout;err=p.stderr
 except subprocess.TimeoutExpired as e:
  code=None;out=e.stdout or b"";err=e.stderr or b""
 (folder/(name+".stdout")).write_bytes(out);(folder/(name+".stderr")).write_bytes(err)
 save(folder/(name+".command.json"),{"binary_sha256":sha(BIN.read_bytes()),"input_sha256":sha(data),"exit":code,"elapsed_seconds":time.time()-started,"environment":{"RUSTUP_TOOLCHAIN":ENV["RUSTUP_TOOLCHAIN"],"TMPDIR":ENV["TMPDIR"]}})
 if code!=0:return None,{"exit":code,"stderr":err.decode(errors="replace")[-2000:]}
 return json.loads(out),None
selected=[
 ("jute-state","jute-notebook/src-tauri/src/state.rs",["repair_object_store_location","is_object_store_location"]),
 ("header-prefix","rest-table-gateway/src/adapter/manifest_adapter.rs",["apply_header_value_prefix"]),
 ("guarded-object-store","src/mcp/datasource/object_store.rs",["repair_object_store_location"]),
 ("guarded-text","src/dag/ai/inputs.rs",["bounded_text"]),
 ("wire-protocol","jute-notebook/src-tauri/src/backend/wire_protocol/driver_zeromq.rs",["from_zmq_payload"]),
]
rows=[]
for key,relative,names in selected:
 folder=ART/"per-file"/key;folder.mkdir(parents=True,exist_ok=True)
 root=folder/"root";target=root/relative;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes((CAPTURE/relative).read_bytes())
 row={"case_id":key,"file":relative,"source_sha256":sha(target.read_bytes()),"scope":"complete original file in isolated graph; declared model profile, not compiled workspace"}
 snap,error=cli(folder,"snapshot",{"command":"snapshot","root":str(root),"source_paths":[relative]})
 if error:row.update(stage="snapshot",error=error);rows.append(row);save(ART/"per-file-results.json",rows);continue
 graph=snap["graph"];sources=[{"file_path":relative,"text":target.read_text()}]
 row.update(graph_content_hash=graph["graph_content_hash"],method_sites=len(snap["method_sites"]["sites"]),macro_sites=len(graph["macro_invocations"]["rows"]),graph_calls=sum(e["relation"]=="calls" for e in graph["edges"]))
 analyzed,error=cli(folder,"analyze",{"command":"analyze","graph":graph,"sources":sources,"build":build(graph)})
 if error:row.update(stage="analyze",error=error);rows.append(row);save(ART/"per-file-results.json",rows);continue
 row["analysis_status"]=analyzed["status"]
 if analyzed["status"]!="collected":
  row.update(stage="analyze",obligation=analyzed.get("obligation"));rows.append(row);save(ART/"per-file-results.json",rows);continue
 facts=analyzed["facts"]
 row.update(bodies=len(facts["bodies"]),unknown_obligations=len(facts["unknown_obligations"]),semantic_operations=len(facts.get("extraction",{}).get("operations",[])))
 row["queries"]=[]
 for name in names:
  symbols=[s for s in graph["symbols"] if s["file_path"]==relative and s["entity_name"]==name]
  assert len(symbols)==1,(key,name,len(symbols))
  body=next((b["body_id"] for b in facts["bodies"] if b.get("definition_stable_symbol_id")==symbols[0]["stable_symbol_id"]),None)
  if body is None:row["queries"].append({"name":name,"outcome":"unmodeled_body"});continue
  values=[v for v in facts["values"] if v["context"]["location"]["body_id"]==body and v["origin"]["kind"]=="parameter"]
  domains=[]
  for value in values:
   kind=value["domain"]["kind"]
   if kind=="u8":domain={"kind":"u8","minimum":0,"maximum":255}
   elif kind=="bool":domain={"kind":"bool","values":[False,True]}
   else:raise ValueError("Unexpected modeled parameter domain "+kind)
   domains.append({"body_id":body,"parameter_index":value["origin"]["index"],"domain":domain})
  req={"command":"solve","graph":graph,"sources":sources,"build":build(graph),"facts":facts,"domains":domains,"limits":{"entry_body_ids":[body],"expected_build_hash":facts["build_hash"],"max_call_depth":0,"max_loop_iterations":0,"max_states":256,"timeout_ms":30000,"persist":True}}
  solved,error=cli(folder,"solve-"+name,req)
  if error:row["queries"].append({"name":name,"error":error});continue
  query=solved["report"]["queries"][0]
  row["queries"].append({"name":name,"outcome":query["outcome"],"raw_status":query.get("response",{}).get("status"),"obligations":len(query["obligations"]),"candidate":query.get("candidate"),"request_identity":query["request_identity"],"solve_id":query.get("response",{}).get("solve_id")})
 row["stage"]="completed";rows.append(row);save(ART/"per-file-results.json",rows)
 print(json.dumps(row),flush=True)
save(ART/"per-file-scan-complete.json",{"files":len(rows),"complete":True})
'''
(ART/"scan-original-files.py").write_text(scan_helper)
scan_stdout=open(ART/"per-file-scan.stdout","wb");scan_stderr=open(ART/"per-file-scan.stderr","wb")
scan_job=subprocess.Popen([sys.executable,str(ART/"scan-original-files.py")],cwd=ART,stdout=scan_stdout,stderr=scan_stderr,env=ENV)
print(json.dumps({"per_file_scan_pid":scan_job.pid,"files":5,"whole_snapshot_failure_retained":True},indent=2))
show("src/mcp/datasource/object_store.rs",1,40)


{
  "per_file_scan_pid": 75186,
  "files": 5,
  "whole_snapshot_failure_retained": true
}

SOURCE src/mcp/datasource/object_store.rs
1: //! Public / private object-store scans (DuckDB httpfs).
2: 
3: use anyhow::{bail, Result};
4: use jute::commands::DatasourceKind;
5: 
6: #[cfg(feature = "datasource-introspect")]
7: use serde::Deserialize;
8: #[cfg(feature = "datasource-introspect")]
9: use std::{
10:     fmt,
11:     path::{Path, PathBuf},
12:     process::Command,
13: };
14: 
15: fn python_string_literal(value: &str) -> String {
16:     serde_json::to_string(value).expect("string literal serializes")
17: }
18: 
19: const OBJECT_STORE_SCHEMES: &[&str] = &[
20:     "https", "http", "s3a", "s3", "gs", "gcs", "abfss", "abfs", "az",
21: ];
22: 
23: #[cfg(feature = "datasource-introspect")]
24: const MANAGED_OBJECT_STORE_PROBE_SCRIPT: &str = r#"
25: import json
26: import sys
27: 
28: def emit(payload):
29:     print(json.dumps(payload, separators=(",", ":")))
30: 
31: expected_version = 

In [11]:
# Explicit analyst-authored site models, separate from automatic Rust semantic extraction.
# They model a valid string made of ASCII 'a' followed by one two-byte scalar 'é'.
segments["guarded-object-store-constant"]=segment("src/mcp/datasource/object_store.rs",19,21)
segments["guarded-object-store"]=segment("src/mcp/datasource/object_store.rs",116,143)
save(ART/"replay-source-segments.json",segments)
def variable(name):return {"kind":"var","name":name}
def integer(value):return {"kind":"int","value":value}
def op(name,*args):return {"kind":"op","op":name,"args":list(args)}
def utf8_model(case,cut,source_key,guarded=False,ascii_only=False):
    prefix=variable("ascii_prefix")
    width=variable("scalar_width")
    length=op("add",prefix,width)
    constraints=[
      {"id":"source_length_guard","expr":op("ge",length,integer(cut))},
      {"id":"cut_inside_scalar_start","expr":op("lt",prefix,integer(cut))},
      {"id":"cut_inside_scalar_end","expr":op("lt",integer(cut),length)},
    ]
    variables=[{"name":"ascii_prefix","type":"int_range","min":0,"max":15},
               {"name":"scalar_width","type":"int_range","min":1 if ascii_only else 2,"max":1 if ascii_only else 2}]
    if guarded:
        variables.append({"name":"get_succeeded","type":"bool"})
        boundary=op("or",op("le",integer(cut),prefix),op("eq",integer(cut),length))
        constraints += [
          {"id":"checked_get_semantics","expr":op("eq",variable("get_succeeded"),boundary)},
          {"id":"source_branch_taken","expr":variable("get_succeeded")},
        ]
    req={"vars":variables,"constraints":constraints,"timeout_ms":30000,"persist":True,
         "include_smt":True,"use_cache":False}
    return {"case_id":case,"cut":cut,"source_segment":source_key,
            "source_binding":{k:v for k,v in segments[source_key].items() if k!="text"},
            "model_kind":"analyst-authored bounded per-site UTF-8 model; not detector-generated semantic facts",
            "assumptions":["Input is ASCII 'a' repeated ascii_prefix times, followed by exactly one scalar of the modeled UTF-8 width.",
                           "No trim-changing whitespace occurs.","Only the identified slice branch is modeled; no whole-project reachability or complete function safety proof."],
            "request":req,"expected_status":"unsat" if guarded or ascii_only else "sat"}
models=[
 utf8_model("jute-location-utf8",8,"repair-object-store"),
 utf8_model("header-prefix-utf8",6,"header-prefix"),
 utf8_model("guarded-get-utf8",8,"guarded-object-store",guarded=True),
 utf8_model("ascii-boundary-control",8,"repair-object-store",ascii_only=True),
]
save(ART/"model-requests.json",models)
print(json.dumps([{"case_id":m["case_id"],"source":m["source_binding"]["path"],"expected_status":m["expected_status"],"model_kind":m["model_kind"]} for m in models],indent=2))
print("Per-file scan exit:",scan_job.poll())
if (ART/"per-file-results.json").exists():print((ART/"per-file-results.json").read_text()[:4000])
print((ART/"per-file-scan.stderr").read_text(errors="replace")[-2000:])


[
  {
    "case_id": "jute-location-utf8",
    "source": "jute-notebook/src-tauri/src/state.rs",
    "expected_status": "sat",
    "model_kind": "analyst-authored bounded per-site UTF-8 model; not detector-generated semantic facts"
  },
  {
    "case_id": "header-prefix-utf8",
    "source": "rest-table-gateway/src/adapter/manifest_adapter.rs",
    "expected_status": "sat",
    "model_kind": "analyst-authored bounded per-site UTF-8 model; not detector-generated semantic facts"
  },
  {
    "case_id": "guarded-get-utf8",
    "source": "src/mcp/datasource/object_store.rs",
    "expected_status": "unsat",
    "model_kind": "analyst-authored bounded per-site UTF-8 model; not detector-generated semantic facts"
  },
  {
    "case_id": "ascii-boundary-control",
    "source": "jute-notebook/src-tauri/src/state.rs",
    "expected_status": "unsat",
    "model_kind": "analyst-authored bounded per-site UTF-8 model; not detector-generated semantic facts"
  }
]
Per-file scan exit: 0
[
  {
    "case_i

In [3]:
println!("Native Rust notebook cell: {}-{}", std::env::consts::ARCH, std::env::consts::OS);
let notebook_rustc = std::process::Command::new("rustc").arg("-vV").output()?;
println!("{}", String::from_utf8_lossy(&notebook_rustc.stdout));


Native Rust notebook cell: aarch64-macos


rustc 1.94.1 (e408947bf 2026-03-25)


binary: rustc


commit-hash: e408947bfd200af42db322daf0fadfe7e26d3bd1


commit-date: 2026-03-25


host: aarch64-apple-darwin


release: 1.94.1


LLVM version: 21.1.8


In [12]:
# Preparation only: write exact captured function bytes for include! in Rust cells.
# No function body, signature, constant, or guard is rewritten.
NATIVE = ART / "native-rust"
NATIVE.mkdir(exist_ok=True)
fragment_sets = {
    "jute-original.rs":["object-store-constant","repair-object-store","object-store-caller"],
    "header-original.rs":["header-prefix"],
    "guarded-original.rs":["guarded-object-store-constant","guarded-object-store"],
    "bounded-original.rs":["guarded-bounded-text"],
}
fragment_manifest=[]
for filename,keys in fragment_sets.items():
    parts=[]
    for key in keys:
        item=segments[key]
        data=(CAPTURE/item["path"]).read_bytes()
        first,last=item["byte_range"]
        exact=data[first:last]
        assert sha(exact)==item["segment_sha256"] and exact.decode()==item["text"]
        parts.append(exact)
    combined=b"\n".join(parts)
    (NATIVE/filename).write_bytes(combined)
    fragment_manifest.append({"file":filename,"sha256":sha(combined),
                              "source_segments":keys,"source_bytes_unchanged":True})
save(ART/"native-fragment-manifest.json",fragment_manifest)
print(json.dumps(fragment_manifest,indent=2))


[
  {
    "file": "jute-original.rs",
    "sha256": "c8c87cabc2ba104957e1be0cac7b7c406b3048fa55c85ea4ec41f4ecf3df1646",
    "source_segments": [
      "object-store-constant",
      "repair-object-store",
      "object-store-caller"
    ],
    "source_bytes_unchanged": true
  },
  {
    "file": "header-original.rs",
    "sha256": "ca54d38ae7c7b15f3dca0b08799ca1236a10e728561bf8692eec1be20a2075af",
    "source_segments": [
      "header-prefix"
    ],
    "source_bytes_unchanged": true
  },
  {
    "file": "guarded-original.rs",
    "sha256": "c8533c0b84202bab5619f953988ec59a6b082a01464ad99cfc940312febbeb54",
    "source_segments": [
      "guarded-object-store-constant",
      "guarded-object-store"
    ],
    "source_bytes_unchanged": true
  },
  {
    "file": "bounded-original.rs",
    "sha256": "fc1a525616fd1dd767bcf27b28bff53827a19561e51161332f1a49f80e1654e2",
    "source_segments": [
      "guarded-bounded-text"
    ],
    "source_bytes_unchanged": true
  }
]


In [4]:
mod nb_jute_original {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/jute-original.rs");
    pub fn classify(value: &str) -> String { is_object_store_location(value).to_string() }
}
mod nb_header_original {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/header-original.rs");
    pub fn apply(prefix: Option<&str>, value: &str) -> String { apply_header_value_prefix(prefix, value.to_owned()) }
}
mod nb_guarded_original {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/guarded-original.rs");
    pub fn repair(value: &str) -> String { repair_object_store_location(value) }
}
mod nb_bounded_original {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/bounded-original.rs");
    pub fn render(value: &str, budget: usize) -> String { format!("{:?}", bounded_text(value, budget)) }
}
fn nb_catch<F: FnOnce() -> String + std::panic::UnwindSafe>(id: &str, input: &str, f: F) -> (String,String,String,String) {
    let (outcome,result) = match std::panic::catch_unwind(f) {
        Ok(value) => ("returned".to_owned(), value),
        Err(payload) => {
            let message = payload.downcast_ref::<String>().cloned()
                .or_else(|| payload.downcast_ref::<&str>().map(|s| s.to_string()))
                .unwrap_or_else(|| "<non-string panic>".to_owned());
            ("panic".to_owned(),message)
        }
    };
    (id.to_owned(),input.to_owned(),outcome,result)
}
fn nb_hex(value: &str) -> String {
    value.as_bytes().iter().map(|b| format!("{:02x}",b)).collect()
}
let nb_previous_hook = std::panic::take_hook();
std::panic::set_hook(Box::new(|_| {}));
let mut nb_native_rows = Vec::new();
nb_native_rows.push(nb_catch("jute-utf8", "aaaaaaaé", || nb_jute_original::classify("aaaaaaaé")));
nb_native_rows.push(nb_catch("header-utf8", "aaaaaé", || nb_header_original::apply(Some("Basic "), "aaaaaé")));
nb_native_rows.push(nb_catch("jute-ascii-control", "aaaaaaaZ", || nb_jute_original::classify("aaaaaaaZ")));
nb_native_rows.push(nb_catch("jute-short-utf8-control", "é", || nb_jute_original::classify("é")));
nb_native_rows.push(nb_catch("jute-uri-control", "s3://bucket/file.csv", || nb_jute_original::classify("s3://bucket/file.csv")));
nb_native_rows.push(nb_catch("header-ascii-control", "abc1234", || nb_header_original::apply(Some("Basic "), "abc1234")));
nb_native_rows.push(nb_catch("header-existing-control", "Basic abc", || nb_header_original::apply(Some("Basic "), "Basic abc")));
nb_native_rows.push(nb_catch("header-none-control", "aaaaaé", || nb_header_original::apply(None, "aaaaaé")));
nb_native_rows.push(nb_catch("guarded-unicode-control", "aaaaaaaé", || nb_guarded_original::repair("aaaaaaaé")));
nb_native_rows.push(nb_catch("guarded-uri-control", "s3:/bucket/file.csv", || nb_guarded_original::repair("s3:/bucket/file.csv")));
nb_native_rows.push(nb_catch("bounded-text-unicode-control", "🦀🦀🦀🦀🦀🦀🦀🦀", || nb_bounded_original::render("🦀🦀🦀🦀🦀🦀🦀🦀", 29)));
std::panic::set_hook(nb_previous_hook);
assert!(nb_native_rows.iter().take(2).all(|r| r.2 == "panic" && r.3.contains("not a char boundary")));
assert!(nb_native_rows.iter().skip(2).all(|r| r.2 == "returned"));
let mut nb_native_tsv = String::from("case_id\tinput_hex\toutcome\tresult_hex\n");
for row in &nb_native_rows {
    println!("{}: {} — {}", row.0, row.2, row.3);
    nb_native_tsv.push_str(&format!("{}\t{}\t{}\t{}\n", row.0, nb_hex(&row.1), row.2, nb_hex(&row.3)));
}
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust-results.tsv", &nb_native_tsv)?;
println!("Verified {} native Rust-cell observations: 2 caught panics and 9 controls.", nb_native_rows.len());


jute-utf8: panic — byte index 8 is not a char boundary; it is inside 'é' (bytes 7..9) of `aaaaaaaé`


header-utf8: panic — byte index 6 is not a char boundary; it is inside 'é' (bytes 5..7) of `aaaaaé`


jute-ascii-control: returned — false


jute-short-utf8-control: returned — false


jute-uri-control: returned — true


header-ascii-control: returned — Basic abc1234


... (2 outputs left out)

guarded-unicode-control: returned — aaaaaaaé


guarded-uri-control: returned — s3://bucket/file.csv


bounded-text-unicode-control: returned — ("\n... (input text truncated)", true)


Verified 11 native Rust-cell observations: 2 caught panics and 9 controls.


In [5]:
let nb_private_root = std::path::PathBuf::from("/private/tmp/spur-notebook-panic-evaluation-20261102/jute-state");
let nb_private_relative = "jute-notebook/src-tauri/src/state.rs";
let nb_private_file = nb_private_root.join(nb_private_relative);
std::fs::create_dir_all(nb_private_file.parent().unwrap())?;
let nb_original_bytes = std::fs::read("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/source-snapshot/jute-notebook/src-tauri/src/state.rs")?;
std::fs::write(&nb_private_file, &nb_original_bytes)?;
assert_eq!(std::fs::read(&nb_private_file)?, nb_original_bytes);
let nb_private_request = format!(r#"{{"command":"snapshot","root":{:?},"source_paths":[{:?}]}}"#, nb_private_root.to_str().unwrap(), nb_private_relative);
let mut nb_detector_child = std::process::Command::new("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
    .current_dir(&nb_private_root)
    .env("RUSTUP_TOOLCHAIN", "1.94.1-aarch64-apple-darwin")
    .env("TMPDIR", "/private/tmp")
    .stdin(std::process::Stdio::piped()).stdout(std::process::Stdio::piped()).stderr(std::process::Stdio::piped())
    .spawn()?;
std::io::Write::write_all(nb_detector_child.stdin.as_mut().unwrap(), nb_private_request.as_bytes())?;
drop(nb_detector_child.stdin.take());
let nb_private_observed = nb_detector_child.wait_with_output()?;
let nb_private_art = std::path::PathBuf::from("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/private-root-check");
std::fs::create_dir_all(&nb_private_art)?;
std::fs::write(nb_private_art.join("snapshot.request.json"), &nb_private_request)?;
std::fs::write(nb_private_art.join("snapshot.stdout"), &nb_private_observed.stdout)?;
std::fs::write(nb_private_art.join("snapshot.stderr"), &nb_private_observed.stderr)?;
println!("Private-root detector exit: {:?}; stdout bytes: {}; stderr: {}",
    nb_private_observed.status.code(), nb_private_observed.stdout.len(), String::from_utf8_lossy(&nb_private_observed.stderr));


Private-root detector exit: Some(0); stdout bytes: 595304; stderr: 


In [13]:
# Correct the input layout: a captured source root nested in the SpurCode Git
# worktree triggered artifact path rebasing. The same original file succeeds
# when the input root/cwd are outside that unrelated repository.
private_scan_helper = scan_helper.replace('folder=ART/"per-file"/key', 'folder=ART/"per-file-private"/key')
private_scan_helper = private_scan_helper.replace('root=folder/"root";target=root/relative',
    'root=Path("/private/tmp/spur-notebook-panic-evaluation-20261102")/key;target=root/relative')
private_scan_helper = private_scan_helper.replace('cwd=folder,env=ENV', 'cwd=root,env=ENV')
private_scan_helper = private_scan_helper.replace('"per-file-results.json"', '"per-file-private-results.json"')
private_scan_helper = private_scan_helper.replace('"per-file-scan-complete.json"', '"per-file-private-scan-complete.json"')
assert private_scan_helper != scan_helper and 'root=folder/"root"' not in private_scan_helper
(ART/"scan-original-files-private-root.py").write_text(private_scan_helper)
private_scan_stdout=open(ART/"per-file-private-scan.stdout","wb")
private_scan_stderr=open(ART/"per-file-private-scan.stderr","wb")
private_scan_job=subprocess.Popen([sys.executable,str(ART/"scan-original-files-private-root.py")],
    cwd="/private/tmp",stdout=private_scan_stdout,stderr=private_scan_stderr,env=ENV)
save(ART/"snapshot-layout-correction.json",{
    "original_failures":6,"error":"missing macro source entry",
    "same_source_positive_control":"private-root-check/snapshot.stdout",
    "same_file_sha256":segments["repair-object-store"]["file_sha256"],
    "correction":"Move exact source inputs/cwd outside unrelated enclosing Git worktree; preserve every original failed run.",
    "detector_source_changed":False,
    "source_interpretation":"Nested-worktree snapshot path/rebasing issue, not a no-panic result or a target compile failure.",
})
print(json.dumps({"private_scan_pid":private_scan_job.pid,"files":5,"source_bytes_changed":False},indent=2))


{
  "private_scan_pid": 90742,
  "files": 5,
  "source_bytes_changed": false
}


In [14]:
# Verify persisted MCP models and prepare Rust constants directly from solver witnesses.
def tool_json(value):
    return value.get("structuredContent") or json.loads(next(c["text"] for c in value["content"] if c["type"]=="text"))
model_receipts=json.loads((ART/"model-receipts.json").read_text())["models"]
solver_witnesses=[]
for item in model_receipts:
    receipt=tool_json(item["reload"]);response=tool_json(item["response"])
    assert receipt["status"]==response["status"]==item["expected_status"]
    for key,value in item["request"].items():
        expected=[{"soft":False,**x} for x in value] if key=="constraints" else value
        assert receipt["request"][key]==expected
    source_binding=item["source_binding"]
    original=(CAPTURE/source_binding["path"]).read_bytes()
    assert sha(original)==source_binding["file_sha256"]
    assert sha(original[slice(*source_binding["byte_range"])])==source_binding["segment_sha256"]
    if receipt["status"]=="sat":
        model=receipt["model"]
        assert model["scalar_width"]==2
        witness="a"*model["ascii_prefix"]+"é"
        assert len(witness.encode())>=item["cut"]
        assert model["ascii_prefix"]<item["cut"]<len(witness.encode())
        solver_witnesses.append({"case_id":item["case_id"],"input":witness,"solve_id":receipt["solve_id"],"cut":item["cut"]})
save(ART/"solver-witnesses.json",solver_witnesses)
rust_constants="pub const WITNESSES: &[(&str,&str,&str)] = &[\n"
for row in solver_witnesses:
    rust_constants += "    ("+", ".join(json.dumps(row[k],ensure_ascii=False) for k in ["case_id","input","solve_id"])+"),\n"
rust_constants += "];\n"
(NATIVE/"solver-witnesses.rs").write_text(rust_constants)
print(json.dumps(solver_witnesses,ensure_ascii=False,indent=2))
print("Corrected per-file scan exit:",private_scan_job.poll())
if (ART/"per-file-private-results.json").exists():
    final_file_rows=json.loads((ART/"per-file-private-results.json").read_text())
    print(json.dumps(final_file_rows,indent=2)[:10000])
print((ART/"per-file-private-scan.stderr").read_text(errors="replace")[-2000:])


[
  {
    "case_id": "jute-location-utf8",
    "input": "aaaaaaaé",
    "solve_id": "sol_7da43826462b4f1c",
    "cut": 8
  },
  {
    "case_id": "header-prefix-utf8",
    "input": "aaaaaé",
    "solve_id": "sol_ed54d0b0e8e0428e",
    "cut": 6
  }
]
Corrected per-file scan exit: 0
[
  {
    "case_id": "jute-state",
    "file": "jute-notebook/src-tauri/src/state.rs",
    "source_sha256": "3bd47d2605d21fe76979c5ba295f6e8167558c6cc4a7811601f0644b4cfb5ec8",
    "scope": "complete original file in isolated graph; declared model profile, not compiled workspace",
    "graph_content_hash": "f2bb17cd8b3a6a67edbe3bd8fdb348188fd283ba14557c3b449e1ddfd88c73bf",
    "method_sites": 363,
    "macro_sites": 69,
    "graph_calls": 507,
    "analysis_status": "unknown",
    "stage": "analyze",
    "obligation": {
      "kind": "cfg",
      "detail": "unevaluated cfg/cfg_attr prevents complete Rust call/body evidence"
    }
  },
  {
    "case_id": "header-prefix",
    "file": "rest-table-gateway/src/adapt

In [6]:
mod nb_model_inputs {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/solver-witnesses.rs");
}
let nb_solver_hook = std::panic::take_hook();
std::panic::set_hook(Box::new(|_| {}));
let mut nb_solver_rows = String::from("case_id\tsolve_id\tinput_hex\toutcome\tpayload_hex\n");
for &(case_id,input,solve_id) in nb_model_inputs::WITNESSES {
    let observed = if case_id == "jute-location-utf8" {
        nb_catch(case_id,input,||nb_jute_original::classify(input))
    } else {
        nb_catch(case_id,input,||nb_header_original::apply(Some("Basic "),input))
    };
    assert_eq!(observed.2,"panic");
    assert!(observed.3.contains("not a char boundary"));
    println!("{} / {} → {}: {}",case_id,solve_id,observed.2,observed.3);
    nb_solver_rows.push_str(&format!("{}\t{}\t{}\t{}\t{}\n",case_id,solve_id,nb_hex(input),observed.2,nb_hex(&observed.3)));
}
std::panic::set_hook(nb_solver_hook);
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/solver-witness-replays.tsv",&nb_solver_rows)?;


jute-location-utf8 / sol_7da43826462b4f1c → panic: byte index 8 is not a char boundary; it is inside 'é' (bytes 7..9) of `aaaaaaaé`


header-prefix-utf8 / sol_ed54d0b0e8e0428e → panic: byte index 6 is not a char boundary; it is inside 'é' (bytes 5..7) of `aaaaaé`


In [7]:
mod nb_matrix_bounded {
    include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-rust/bounded-original.rs");
    pub fn render(value: &str,budget: usize) -> (String,bool) { bounded_text(value,budget) }
}
let nb_matrix_hook = std::panic::take_hook();
std::panic::set_hook(Box::new(|_| {}));
let mut nb_matrix_rows = String::from("case_id\tfamily\tinput_hex\tbudget\toutcome\tresult_hex\tpredicted_panic\n");
let mut nb_matrix_total = 0usize;
let mut nb_matrix_panics = 0usize;
let mut nb_matrix_source_panics = [0usize;3];
for ascii_prefix in 0..16usize {
    for scalar in ["a","é","界","🦀"] {
        let input = format!("{}{}bbbbbbbb","a".repeat(ascii_prefix),scalar);
        let boundary_bad = |cut: usize| input.len()>=cut && !input.is_char_boundary(cut);
        let jute_predicted = [8usize,7,6,5,4].into_iter().any(boundary_bad);
        let header_predicted = boundary_bad(6);
        let cases = [
            nb_catch("jute-matrix",&input,||nb_jute_original::classify(&input)),
            nb_catch("header-matrix",&input,||nb_header_original::apply(Some("Basic "),&input)),
            nb_catch("guarded-matrix",&input,||nb_guarded_original::repair(&input)),
        ];
        for (family,observed,predicted) in [
            (0usize,&cases[0],jute_predicted),
            (1usize,&cases[1],header_predicted),
            (2usize,&cases[2],false),
        ] {
            assert_eq!(observed.2=="panic",predicted);
            nb_matrix_total+=1;
            nb_matrix_panics+=usize::from(observed.2=="panic");
            nb_matrix_source_panics[family]+=usize::from(observed.2=="panic");
            nb_matrix_rows.push_str(&format!("{}-{}-{}\t{}\t{}\t\t{}\t{}\t{}\n",
                observed.0,ascii_prefix,scalar.len(),["jute","header","guarded"][family],
                nb_hex(&input),observed.2,nb_hex(&observed.3),predicted));
        }
        for budget in 0..65usize {
            let observed = std::panic::catch_unwind(||nb_matrix_bounded::render(&input,budget));
            let (outcome,result) = match observed {
                Ok((text,truncated)) => {
                    assert!(text.len()<=budget);
                    assert_eq!(truncated,input.len()>budget);
                    if !truncated { assert_eq!(text,input); }
                    ("returned",text)
                },
                Err(_) => ("panic",String::from("unexpected bounded-text panic")),
            };
            assert_eq!(outcome,"returned");
            nb_matrix_total+=1;
            nb_matrix_rows.push_str(&format!("bounded-{}-{}-{}\tbounded_text\t{}\t{}\t{}\t{}\tfalse\n",
                ascii_prefix,scalar.len(),budget,nb_hex(&input),budget,outcome,nb_hex(&result)));
        }
    }
}
std::panic::set_hook(nb_matrix_hook);
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-utf8-matrix.tsv",&nb_matrix_rows)?;
println!("Rust-cell matrix: {} calls; {} caught source panics; per family {:?}; all predicted outcomes matched.",nb_matrix_total,nb_matrix_panics,nb_matrix_source_panics);
println!("Kernel cfg: debug_assertions={}, panic_unwind={}, pointer_width={}",cfg!(debug_assertions),cfg!(panic="unwind"),usize::BITS);


Rust-cell matrix: 4352 calls; 24 caught source panics; per family [18, 6, 0]; all predicted outcomes matched.


Kernel cfg: debug_assertions=true, panic_unwind=true, pointer_width=64


In [15]:
# One corrected broad snapshot: input bytes outside the unrelated SpurCode Git tree.
# Source-method collection is limited to project-owned source directories; fixture/vendor
# files stay in the archived inventory and are not relabeled as production coverage.
PRIVATE_WHOLE = Path("/private/tmp/spur-notebook-panic-evaluation-20261102/whole-project")
for record in source_rows:
    target_file=PRIVATE_WHOLE/record["path"]
    target_file.parent.mkdir(parents=True,exist_ok=True)
    data=(CAPTURE/record["path"]).read_bytes()
    assert sha(data)==record["sha256"]
    target_file.write_bytes(data)
BROAD = ART/"whole-private"
BROAD.mkdir(exist_ok=True)
owned_rust=[r["path"] for r in source_rows if r["rust"] and r["path"].startswith(owned_prefixes)]
save(BROAD/"snapshot.request.json",{"command":"snapshot","root":str(PRIVATE_WHOLE),"source_paths":owned_rust})
save(BROAD/"input-scope.json",{
    "copied_files":len(source_rows),"all_captured_rust_files":sum(r["rust"] for r in source_rows),
    "requested_project_owned_rust_files":len(owned_rust),"requested_paths":owned_rust,
    "meaning":"Physical source inventory and direct sites, not Cargo feature-expanded or compiled execution coverage."
})
print(json.dumps({"copied_input_files":len(source_rows),"requested_owned_rust_files":len(owned_rust)},indent=2))


{
  "copied_input_files": 721,
  "requested_owned_rust_files": 522
}


In [8]:
let nb_whole_art = std::path::PathBuf::from("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/whole-private");
let mut nb_whole_child = std::process::Command::new("/private/tmp/r8g-bd-3d9-b2ae97ed-target/debug/examples/panic_evidence")
    .current_dir("/private/tmp/spur-notebook-panic-evaluation-20261102/whole-project")
    .env("RUSTUP_TOOLCHAIN","1.94.1-aarch64-apple-darwin").env("TMPDIR","/private/tmp")
    .stdin(std::fs::File::open(nb_whole_art.join("snapshot.request.json"))?)
    .stdout(std::fs::File::create(nb_whole_art.join("snapshot.stdout"))?)
    .stderr(std::fs::File::create(nb_whole_art.join("snapshot.stderr"))?)
    .spawn()?;
let nb_whole_pid = nb_whole_child.id();
std::fs::write(nb_whole_art.join("pid.txt"),format!("{}",nb_whole_pid))?;
println!("Corrected broad detector snapshot launched from Rust cell; PID {}.",nb_whole_pid);


Corrected broad detector snapshot launched from Rust cell; PID 5497.


In [16]:
CREATE OR REPLACE TEMP TABLE native_panic_matrix AS
SELECT * FROM read_csv('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-utf8-matrix.tsv', delim='\t',header=true,all_varchar=true);
COPY native_panic_matrix TO '/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/native-utf8-matrix.parquet' (FORMAT PARQUET);
SELECT family, count(*) AS calls,
       count(*) FILTER (WHERE outcome='panic') AS caught_panics,
       bool_and((outcome='panic')=(predicted_panic='true')) AS outcomes_match_site_model
FROM native_panic_matrix GROUP BY family ORDER BY family;

family,calls,caught_panics,outcomes_match_site_model
bounded_text,4160,0,True
guarded,64,0,True
header,64,6,True
jute,64,18,True


In [11]:
let nb_whole_status = nb_whole_child.try_wait()?;
println!("Broad snapshot status: {:?}; output bytes: {}",
    nb_whole_status.map(|s|s.code()), std::fs::metadata(nb_whole_art.join("snapshot.stdout"))?.len());
if let Some(status) = nb_whole_status {
    std::fs::write(nb_whole_art.join("exit.json"),format!(r#"{{"exit":{:?},"pid":{}}}"#,status.code().unwrap_or(-1),nb_whole_pid))?;
    println!("Broad snapshot stderr: {}",std::fs::read_to_string(nb_whole_art.join("snapshot.stderr"))?);
}


Broad snapshot status: None; output bytes: 0


()

In [17]:
# Export exact automatic solver queries for independent endpoint cross-checks.
automatic_queries=[]
for key in ["guarded-text","wire-protocol"]:
    folder=ART/"per-file-private"/key
    for solve_file in sorted(folder.glob("solve-*.stdout")):
        report=json.loads(solve_file.read_text())["report"]
        for query in report["queries"]:
            automatic_queries.append({
                "case_id":key,"request":query["request"],
                "request_identity":query["request_identity"],"build_hash":report["build_hash"],
                "fact_digest":report["fact_digest"],"sources":report["sources"],
                "original_cli_solve_id":query["response"]["solve_id"],
                "original_cli_status":query["response"]["status"],
                "analysis_outcome":query["outcome"],"obligations":query["obligations"],
            })
save(ART/"automatic-query-requests.json",automatic_queries)
print(json.dumps([{"case_id":q["case_id"],"outcome":q["analysis_outcome"],"obligations":len(q["obligations"])} for q in automatic_queries],indent=2))
# Check target source stability without changing any project file.
drift=[]
for record in source_rows:
    live=PROJECT/record["path"]
    if not live.is_file() or sha(live.read_bytes())!=record["sha256"]:drift.append(record["path"])
after={"head":git("rev-parse","HEAD").decode().strip(),"captured_files_checked":len(source_rows),
       "source_drift_paths":drift,"no_edits_by_experiment":True}
save(ART/"project-state-after.json",after)
print(json.dumps(after,indent=2))


[
  {
    "case_id": "guarded-text",
    "outcome": "incomplete_semantics",
    "obligations": 8
  },
  {
    "case_id": "wire-protocol",
    "outcome": "incomplete_semantics",
    "obligations": 12
  }
]
{
  "head": "7fb364638575da96d46a68c5ec6af0215fe97300",
  "captured_files_checked": 721,
  "source_drift_paths": [],
  "no_edits_by_experiment": true
}


In [13]:
let nb_snapshot_elapsed = std::fs::metadata(nb_whole_art.join("snapshot.stdout"))?.created()?.elapsed()?.as_secs();
let nb_process_observation = std::process::Command::new("ps").args(["-p",&nb_whole_pid.to_string(),"-o","pid,etime,time,rss,stat"]).output()?;
println!("Broad snapshot elapsed seconds: {}",nb_snapshot_elapsed);
println!("{}",String::from_utf8_lossy(&nb_process_observation.stdout));
let nb_latest_snapshot_status = nb_whole_child.try_wait()?;
if nb_latest_snapshot_status.is_none() && nb_snapshot_elapsed > 600 {
    nb_whole_child.kill()?;
    let status = nb_whole_child.wait()?;
    std::fs::write(nb_whole_art.join("exit.json"),format!(r#"{{"exit":-1,"pid":{},"elapsed_seconds":{},"terminated_by_experiment":true,"budget_seconds":600}}"#,nb_whole_pid,nb_snapshot_elapsed))?;
    println!("Stopped broad snapshot after the experiment's 600-second observation budget: {:?}",status);
} else if let Some(status)=nb_latest_snapshot_status {
    std::fs::write(nb_whole_art.join("exit.json"),format!(r#"{{"exit":{},"pid":{},"elapsed_seconds":{},"terminated_by_experiment":false}}"#,status.code().unwrap_or(-1),nb_whole_pid,nb_snapshot_elapsed))?;
    println!("Broad snapshot completed: {:?}; output bytes {}",status.code(),std::fs::metadata(nb_whole_art.join("snapshot.stdout"))?.len());
} else {
    println!("Broad snapshot still running within observation budget.");
}


Broad snapshot elapsed seconds: 645


  PID ELAPSED      TIME    RSS STAT


 5497   10:45  10:40.38  54288 R   


Stopped broad snapshot after the experiment's 600-second observation budget: ExitStatus(unix_wait_status(9))


()

In [18]:
# Verify the independent worker's source bindings and read the review audit from
# the authoritative tracker database in read-only mode (the br CLI is unavailable).
import sqlite3
worker_commit="b4b0de765"
review_dir=ART/"independent-review"
review_dir.mkdir(exist_ok=True)
for name in ["REPORT.md","candidates.json"]:
    relative="docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/independent-review/"+name
    data=subprocess.check_output(["git","-C",str(DETECTOR_SOURCE),"show",worker_commit+":"+relative])
    (review_dir/name).write_bytes(data)
worker_ledger=json.loads((review_dir/"candidates.json").read_text())
manifest_by_path={r["path"]:r for r in source_rows}
for relative,binding in worker_ledger["sources"].items():
    saved=manifest_by_path[relative]
    assert all(binding[key]==saved[key] for key in ["bytes","sha256","git_blob_oid"])
    assert sha((CAPTURE/relative).read_bytes())==binding["sha256"]
assert sha((ART/"source-manifest.json").read_bytes())==worker_ledger["source_capture"]["manifest_sha256"]
assert len(worker_ledger["candidates"])==7 and len(worker_ledger["guard_protected_controls"])==3
for row in worker_ledger["candidates"]+worker_ledger["guard_protected_controls"]:
    first,last=row["function_lines"]
    assert 1<=first<=last<=len((CAPTURE/row["path"]).read_text().splitlines())
review_verification={"commit":worker_commit,"candidates":7,"protected_controls":3,"source_files_verified":len(worker_ledger["sources"]),"all_source_bindings_match":True}
beads=Path("/Users/kevintruong/project/SpurCode/.beads")
tracker_observations=[]
for database in sorted(beads.glob("*.db")):
    if not database.is_file():continue
    conn=sqlite3.connect(database.resolve().as_uri()+"?mode=ro",uri=True)
    tables={row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if "comments" in tables:
        columns=[row[1] for row in conn.execute("PRAGMA table_info(comments)")]
        if "issue_id" in columns:
            conn.row_factory=sqlite3.Row
            comments=[dict(row) for row in conn.execute("SELECT * FROM comments WHERE issue_id = ?",("bd-3qk",))]
            for comment in comments:
                body=comment.get("text",comment.get("body",""))
                if "[[spur-audit v1]]" in body:
                    tracker_observations.append({"database":str(database),"id":comment.get("id"),"body":body,"body_sha256":sha(body.encode())})
    conn.close()
save(ART/"worker-review-verification.json",review_verification)
save(ART/"worker-tracker-audits.json",tracker_observations)
print(json.dumps(review_verification,indent=2))
print("Durable tracker audit summaries:",json.dumps([{"id":c["id"],"prefix":c["body"][:260],"body_sha256":c["body_sha256"]} for c in tracker_observations],indent=2))


{
  "commit": "b4b0de765",
  "candidates": 7,
  "protected_controls": 3,
  "source_files_verified": 11,
  "all_source_bindings_match": true
}
Durable tracker audit summaries: [
  {
    "id": 1971,
    "prefix": "[[spur-audit v1]]\n{\"kind\":\"task-spec\",\"task_id\":\"NB-AUDIT\",\"context_files\":[],\"planned_write_files\":[\"docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/independent-review/REPORT.md\",\"docs/research/artifacts/2026-11-02-spur-notebook-panic-evalua",
    "body_sha256": "27576bcd624011cc1d1e0370833dda9fb6baef3f2d606b08aad2fcadd19e5a37"
  },
  {
    "id": 1974,
    "prefix": "[[spur-audit v1]]\n{\"kind\":\"dispatch\",\"delegation_id\":\"6cd8e56a98a147f9\",\"worker\":\"codex\",\"attempt\":1}",
    "body_sha256": "5468c286c882c1c33c963eb7befd24c896d00017a3c579b186ae87f6718cd5f7"
  },
  {
    "id": 1975,
    "prefix": "[[spur-audit v1]]\n{\"kind\":\"worker-started\",\"delegation_id\":\"6cd8e56a98a147f9\",\"worker_branch\":\"spur/worker/v2/codex/99eaf8bce1

In [21]:
verification_run=subprocess.run([sys.executable,str(ART/"verify.py")],capture_output=True,text=True,timeout=60)
assert verification_run.returncode==0,verification_run.stderr
verification=json.loads(verification_run.stdout)
save(ART/"verification.json",verification)
print(json.dumps(verification,indent=2))


{
  "all_checks_passed": true,
  "source_files": 721,
  "rust_files": 689,
  "target_source_drift": 0,
  "corrected_original_files": 5,
  "captured_method_sites": 1846,
  "captured_macro_sites": 564,
  "automatic_cfg_unknown_files": 3,
  "automatic_incomplete_queries": 2,
  "automatic_confirmed_witnesses": 0,
  "manual_source_models": 4,
  "fresh_reloaded_mcp_receipts": 6,
  "smoke_native_calls": 11,
  "smoke_panics": 2,
  "solver_witness_replays": 2,
  "native_matrix_calls": 4352,
  "native_matrix_panics": 24,
  "distinct_reproduced_source_helpers": 2,
  "independent_conditional_candidates": 7,
  "independent_guard_controls": 3,
  "broad_snapshot_observed_seconds": 645,
  "broad_snapshot_completed": false,
  "sealed_payloads_verified": 807,
  "scope": "Exact-function Rust-cell reproductions and explicit analyst models; no automatic whole-project panic-detection or application-reachability claim."
}


In [20]:
# Archive immutable large inputs/outputs without source-tree duplication in Git.
# Only task-owned evidence is read. Private solver stores and caches are excluded.
import tarfile, gzip, io
bundle_dirs=["source-snapshot","detector","per-file","per-file-private","private-root-check","whole-private"]
bundle_rows=[]
for directory in bundle_dirs:
    for current,dirs,files in os.walk(ART/directory):
        dirs[:]=sorted(d for d in dirs if d not in {".spur","target","identity-target","cargo-home","__pycache__"})
        for name in sorted(files):
            path=Path(current)/name
            assert path.is_file() and not path.is_symlink()
            content=path.read_bytes()
            bundle_rows.append({"path":path.relative_to(ART).as_posix(),"bytes":len(content),"sha256":sha(content)})
bundle_rows.sort(key=lambda x:x["path"])
with open(ART/"evidence.tar.gz","wb") as raw:
    with gzip.GzipFile(fileobj=raw,mode="wb",mtime=0) as compressed:
        with tarfile.open(fileobj=compressed,mode="w|") as archive:
            for row in bundle_rows:
                content=(ART/row["path"]).read_bytes()
                assert sha(content)==row["sha256"]
                info=tarfile.TarInfo(row["path"])
                info.size=len(content);info.mode=0o644;info.mtime=0;info.uid=0;info.gid=0
                archive.addfile(info,io.BytesIO(content))
save(ART/"bundle-manifest.json",bundle_rows)
bundle_summary={"files":len(bundle_rows),"uncompressed_bytes":sum(r["bytes"] for r in bundle_rows),
                "archive_bytes":(ART/"evidence.tar.gz").stat().st_size,
                "archive_sha256":sha((ART/"evidence.tar.gz").read_bytes()),
                "excluded":[".spur","target","identity-target","cargo-home","__pycache__"]}
save(ART/"bundle-summary.json",bundle_summary)
print(json.dumps(bundle_summary,indent=2))


{
  "files": 807,
  "uncompressed_bytes": 22152162,
  "archive_bytes": 3517383,
  "archive_sha256": "19eeff86bd25c69c2724b5173eece7580312b6c2c180fdb772c7eb6f8f7fd1ad",
  "excluded": [
    ".spur",
    "target",
    "identity-target",
    "cargo-home",
    "__pycache__"
  ]
}


## Results from direct Rust cells

The notebook ran native Rust under **Rust 1.94.1**, with unwind panics, debug assertions and 64-bit pointers. Exact captured function bodies and constants are included from `native-rust/*.rs`; wrappers call them without rewriting their logic.

| Confirmed helper-level failure | Concrete input | Native result |
|---|---|---|
| Jute `repair_object_store_location`, through unchanged `is_object_store_location` | `aaaaaaaé` | Caught panic: byte index **8** is inside é |
| `apply_header_value_prefix` | Prefix `Some("Basic ")`, value `aaaaaé` | Caught panic: byte index **6** is inside é |

The smoke cell observed **2 caught panics and 9 normal controls**. Two source-bound SAT witnesses were then replayed from persisted MCP model results in Rust cells and reproduced both payloads. The larger finite matrix ran **4,352 calls**: 18 panic cases at the Jute helper, 6 at the header helper, and none in the existing guarded object-store or bounded-text controls. These are **two reproduced source helpers**, not 24 distinct bugs or a whole-application execution result.

## What the automatic detector established

The corrected five-file snapshots captured **1,846 method sites, 564 macro sites and 2,668 Calls**. Original-file semantic analysis returned **3 configuration unknowns** and **2 incomplete queries**, with no automatic witness. The incomplete queries have raw UNSAT responses and **8 / 12 unresolved obligations**; they are not safety results.

The UTF-8 site models were explicitly written by the analyst from source preconditions. Their 2 SAT / 2 UNSAT results and six fresh reloaded MCP receipts (including two automatic-query cross-checks) are preserved separately. **Native reproduction works; automatic discovery of these string-boundary panics is not implemented by the current bounded semantic subset.**

## Source scope and practical limits

- The target source capture contains 689 Rust files at HEAD `7fb364638575da96d46a68c5ec6af0215fe97300`; all 721 captured source/metadata files matched the live checkout at the recorded stability check.
- Exact source extraction changes compilation context. No full Tauri application, credential state, or kernel transport was exercised.
- Independent review checked seven conditional production candidates and three guarded controls. Only the two Unicode helpers above have native reproductions here. Malformed transport/Arrow inputs, arithmetic profiles, theoretical exhaustion and lock poisoning remain conditional.
- Six initial nested-worktree snapshot failures are preserved. The same bytes succeed from private input roots, exposing a path-rebinding integration seam.
- A corrected broad source snapshot requested 522 owned Rust files but did not finish before the observation budget; it was stopped at 645 seconds after exceeding the 600-second threshold. Whole-project coverage remains unestablished.
- The next detector work should address per-body cfg handling, typed strings/usize/range boundaries and checked-get guards, robust source-root identities, and broad extraction cost.

[Full report and rerun instructions](docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/README.md) · [Independent review](docs/research/artifacts/2026-11-02-spur-notebook-panic-evaluation/independent-review/REPORT.md)

The notebook is an executed research chronology. Re-run the committed Rust fragments with the Rust probe, smoke, witness and matrix cells for direct reproduction. A new full capture requires a fresh output directory; preserve this recorded run.
